In [ ]:
## Notebook 02 — Data Preparation

### Objectif :
Preparer les donnees pour la modelisation en suivant les etapes CRISP-DM Phase 3.

### Etapes :
1. Chargement des donnees nettoyees de l'EDA
2. Imputation des valeurs manquantes
3. Encodage des variables categorielles (One-Hot Encoding)
4. Normalisation des variables numeriques (StandardScaler)
5. Traitement du desequilibre de classes (SMOTE)
6. Decoupage train/test (80%/20% stratifie)
7. Sauvegarde des donnees preparees

In [1]:
# Cellule 1 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Chargement
df = pd.read_csv(r'C:\Users\merie\churn-prediction\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv')

df = df.rename(columns={
    'tenure': 'mois_abonnement',
    'PhoneService': 'service_base',
    'MultipleLines': 'profils_partages',
    'InternetService': 'type_connexion',
    'OnlineSecurity': 'controle_parental',
    'OnlineBackup': 'sauvegarde_contenu',
    'DeviceProtection': 'protection_compte',
    'TechSupport': 'support_technique',
    'StreamingTV': 'streaming_tv',
    'StreamingMovies': 'streaming_films',
    'Contract': 'type_contrat',
    'PaperlessBilling': 'facturation_en_ligne',
    'PaymentMethod': 'mode_paiement',
    'MonthlyCharges': 'montant_mensuel',
    'TotalCharges': 'montant_total',
    'Churn': 'churn'
})

df['montant_total'] = pd.to_numeric(df['montant_total'], errors='coerce')

print("Donnees chargees !")
print(f"Dimensions : {df.shape}")

Donnees chargees !
Dimensions : (7043, 21)


In [ ]:
## Etape 1 — Imputation des valeurs manquantes

### Pourquoi ?
Les 11 valeurs manquantes de montant_total correspondent a des clients
avec 0 mois d'abonnement. Leur montant total est logiquement 0.

### Choix technique :
On impute par 0 et non par la mediane car ces clients n'ont
pas encore ete factures — imputer la mediane serait incorrect metier.

In [2]:
# Cellule 2 — Imputation des valeurs manquantes
print(f"Valeurs manquantes avant : {df['montant_total'].isnull().sum()}")

df['montant_total'] = df['montant_total'].fillna(0)

print(f"Valeurs manquantes apres : {df['montant_total'].isnull().sum()}")
print("Imputation terminee !")

Valeurs manquantes avant : 11
Valeurs manquantes apres : 0
Imputation terminee !


In [ ]:
## Etape 2 — Encodage des variables categorielles

### Pourquoi ?
Les algorithmes de Machine Learning ne peuvent pas traiter
des variables textuelles directement. On doit les convertir en nombres.

### Choix technique :
- One-Hot Encoding avec pd.get_dummies pour les variables nominales
- La variable cible churn est encodee en binaire (Yes=1, No=0)
- La colonne customerID est supprimee car non predictive

In [3]:
# Cellule 3 — Encodage
# Supprimer customerID
df = df.drop(columns=['customerID'])

# Encoder la variable cible
df['churn'] = df['churn'].map({'Yes': 1, 'No': 0})

# Separer features et cible
X = df.drop(columns=['churn'])
y = df['churn']

# One-Hot Encoding
X = pd.get_dummies(X, drop_first=True)

print(f"Dimensions apres encodage : {X.shape}")
print(f"Nombre de features : {X.shape[1]}")
print(f"\nApercu des colonnes :")
print(X.columns.tolist())

Dimensions apres encodage : (7043, 30)
Nombre de features : 30

Apercu des colonnes :
['SeniorCitizen', 'mois_abonnement', 'montant_mensuel', 'montant_total', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'service_base_Yes', 'profils_partages_No phone service', 'profils_partages_Yes', 'type_connexion_Fiber optic', 'type_connexion_No', 'controle_parental_No internet service', 'controle_parental_Yes', 'sauvegarde_contenu_No internet service', 'sauvegarde_contenu_Yes', 'protection_compte_No internet service', 'protection_compte_Yes', 'support_technique_No internet service', 'support_technique_Yes', 'streaming_tv_No internet service', 'streaming_tv_Yes', 'streaming_films_No internet service', 'streaming_films_Yes', 'type_contrat_One year', 'type_contrat_Two year', 'facturation_en_ligne_Yes', 'mode_paiement_Credit card (automatic)', 'mode_paiement_Electronic check', 'mode_paiement_Mailed check']


In [ ]:
## Etape 3 — Normalisation des variables numeriques

### Pourquoi ?
Les variables numeriques ont des echelles tres differentes :
- mois_abonnement : 0 a 72
- montant_mensuel : 18 a 119
- montant_total : 0 a 8684

Sans normalisation, les algorithmes comme la Regression Logistique
sont biaises vers les variables a grande echelle.

### Choix technique :
StandardScaler centre les donnees (moyenne=0, ecart-type=1).
Applique uniquement sur les variables numeriques continues.

In [4]:
# Cellule 4 — Normalisation
scaler = StandardScaler()

cols_numeriques = ['mois_abonnement', 'montant_mensuel', 'montant_total']

X[cols_numeriques] = scaler.fit_transform(X[cols_numeriques])

print("Normalisation terminee !")
print(f"\nMoyennes apres normalisation (doivent etre ~0) :")
print(X[cols_numeriques].mean().round(4))
print(f"\nEcarts-types apres normalisation (doivent etre ~1) :")
print(X[cols_numeriques].std().round(4))

Normalisation terminee !

Moyennes apres normalisation (doivent etre ~0) :
mois_abonnement   -0.0
montant_mensuel   -0.0
montant_total     -0.0
dtype: float64

Ecarts-types apres normalisation (doivent etre ~1) :
mois_abonnement    1.0001
montant_mensuel    1.0001
montant_total      1.0001
dtype: float64


In [ ]:
## Etape 4 — Decoupage Train/Test et SMOTE

### Pourquoi decouper d'abord ?
On applique SMOTE uniquement sur les donnees d'entrainement.
Appliquer SMOTE avant le decoupage causerait du data leakage —
des donnees synthetiques du train se retrouveraient dans le test,
ce qui biaiserait les resultats d'evaluation.

### Choix techniques :
- Decoupage 80%/20% stratifie pour conserver le ratio churn dans chaque split
- SMOTE genere des exemples synthetiques de la classe minoritaire (churn=1)
  uniquement sur X_train pour equilibrer les classes

In [5]:
# Cellule 5 — Train/Test split + SMOTE
# Decoupage stratifie
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {X_train.shape[0]} lignes")
print(f"Test : {X_test.shape[0]} lignes")
print(f"\nDistribution churn dans train avant SMOTE :")
print(y_train.value_counts())

# SMOTE sur train uniquement
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"\nDistribution churn dans train apres SMOTE :")
print(y_train_sm.value_counts())
print(f"\nTrain apres SMOTE : {X_train_sm.shape[0]} lignes")

Train : 5634 lignes
Test : 1409 lignes

Distribution churn dans train avant SMOTE :
churn
0    4139
1    1495
Name: count, dtype: int64

Distribution churn dans train apres SMOTE :
churn
0    4139
1    4139
Name: count, dtype: int64

Train apres SMOTE : 8278 lignes


## Etape 4 — Decoupage Train/Test et SMOTE

### 1. Pourquoi decouper en Train/Test ?

Le principe est similaire a un examen scolaire :
- Le professeur enseigne sur certains exercices (Train)
- Il evalue sur des exercices jamais vus (Test)

On divise nos 7043 clients :
- Train (80%) = 5634 clients  → le modele apprend sur ces donnees
- Test (20%) = 1409 clients   → on evalue le modele sur ces donnees

Le parametre "stratify=y" garantit que le ratio 73%/27% 
est conserve dans les deux groupes.

### 2. Pourquoi SMOTE ?

Sur le train on a 4139 Non-Churn vs 1495 Churn.
Ce desequilibre pose un probleme : le modele va apprendre
a dire "Non-Churn" presque tout le temps car c'est la majorite,
ce qui le rend inutile pour detecter les vrais churners.

SMOTE (Synthetic Minority Oversampling Technique) cree des clients
fictifs qui ressemblent aux churners existants pour equilibrer :
- Avant SMOTE : 4139 Non-Churn vs 1495 Churn
- Apres SMOTE : 4139 Non-Churn vs 4139 Churn

### 3. Pourquoi SMOTE uniquement sur le Train ?

Si on appliquait SMOTE sur tout le dataset avant de decouper,
des clients fictifs crees pourraient se retrouver dans le Test.
Le modele serait evalue sur des donnees qu'il a indirectement
vues pendant l'entrainement → resultats faussement optimistes.
C'est ce qu'on appelle le DATA LEAKAGE — a eviter absolument.



In [6]:
# Cellule 6 — Sauvegarde
import os

X_train_sm.to_csv(r'C:\Users\merie\churn-prediction\data\processed\X_train.csv', index=False)
X_test.to_csv(r'C:\Users\merie\churn-prediction\data\processed\X_test.csv', index=False)
y_train_sm.to_csv(r'C:\Users\merie\churn-prediction\data\processed\y_train.csv', index=False)
y_test.to_csv(r'C:\Users\merie\churn-prediction\data\processed\y_test.csv', index=False)

print("Donnees sauvegardees dans data/processed/ !")
print(f"\nFichiers crees :")
for f in os.listdir(r'C:\Users\merie\churn-prediction\data\processed'):
    print(f"  - {f}")

Donnees sauvegardees dans data/processed/ !

Fichiers crees :
  - X_test.csv
  - X_train.csv
  - y_test.csv
  - y_train.csv
